In [87]:
import os
# change working directory to project root
os.chdir("/Users/yensydney/Desktop/pstat197/module-2-claims-data-table-10")
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras.layers import TextVectorization, Dense, Dropout
from tensorflow.keras import Sequential

In [88]:
data_dir = "data"
rdata_files = ["claims-clean-example.RData", "claims-clean.RData", "claims-raw.RData"]

# pick the first file that exists
selected_file = None
for f in rdata_files:
    if os.path.exists(os.path.join(data_dir, f)):
        selected_file = f
        break

if selected_file is None:
    raise FileNotFoundError("No RData files found in data folder")

selected_path = os.path.join(data_dir, selected_file)
print("Loading:", selected_path)

rdata = pyreadr.read_r(selected_path)
print("Objects inside:", list(rdata.keys()))

# pick first object
df = rdata[next(iter(rdata.keys()))]
print(df.head())
print(df.columns)

Loading: data/claims-clean-example.RData
Objects inside: ['claims_clean']
   neo_search_transaction_id  neo_search_subject_id  \
0                 12395162.0             11497914.0   
1                 12394582.0             11499214.0   
2                 12384260.0             11485861.0   
3                 12363093.0             11481705.0   
4                 12376392.0             11488536.0   

                                        original_url  \
0  http://hosting-22647.tributes.com/obituary/sho...   
1  https://www.localcrimenews.com/welcome/arrest/...   
2  https://www.bustedmugshots.com/florida/miami-d...   
3  https://www.policearrests.com/florida-arrest-r...   
4  https://www.bustedmugshots.com/tennessee/memph...   

                                            text_tmp  \
0  <!DOCTYPE html PUBLIC "-//W3C//DTD XHTML 1.0 S...   
1  <!DOCTYPE html>\n<html lang="en">\n<head>\n   ...   
2  <!DOCTYPE html>\n<html>\n<head>\n<meta charset...   
3  <!DOCTYPE HTML>\n<html lang="en

In [89]:
# automatically find columns
text_col = next((c for c in df.columns if "text_clean" in c.lower()), None)
label_col = next((c for c in df.columns if "bclass" in c.lower()), None)
id_col = next((c for c in df.columns if c == ".id"), None)

print("Text column:", text_col)
print("Label column:", label_col)
print("ID column:", id_col)

# convert to proper formats
texts = df[text_col].astype(str).tolist()
labels = df[label_col].astype("category").cat.codes.values  # convert to 0/1
ids = df[id_col].tolist() if id_col else list(range(len(texts)))

print("Number of examples:", len(texts))


Text column: text_clean
Label column: bclass
ID column: .id
Number of examples: 2140


In [101]:
X_train, X_test, y_train, y_test = train_test_split(
    texts, labels, test_size=0.2, random_state=42, stratify=labels
)

X_train_split, X_val, y_train_split, y_val = train_test_split(
    X_train, y_train, test_size=0.3, random_state=42, stratify=y_train
)

In [102]:
# Vectorization
max_tokens = 10000
max_len = 100

vectorize_layer = TextVectorization(
    max_tokens=max_tokens,
    output_mode='int',
    output_sequence_length=max_len
)

# Adapt only on training data
vectorize_layer.adapt(np.array(X_train_split))

# Convert text → integer sequences
X_train_seq = vectorize_layer(np.array(X_train_split))
X_val_seq = vectorize_layer(np.array(X_val))
X_test_seq = vectorize_layer(np.array(X_test))

In [103]:
# Build model
vocab_size = max_tokens + 1
embedding_dim = 16

model = Sequential([
    Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=max_len),
    GlobalAveragePooling1D(),
    Dense(16, activation='relu'),
    Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/keras/src/layers/core/embedding.py:86: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [104]:
# Train
history = model.fit(
    X_train_seq,
    np.array(y_train_split),
    validation_data=(X_val_seq, np.array(y_val)),
    epochs=5,
    batch_size=32
)

Epoch 1/5
38/38 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.5592 - loss: 0.6899 - val_accuracy: 0.6576 - val_loss: 0.6698
Epoch 2/5
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.6278 - loss: 0.6663 - val_accuracy: 0.6946 - val_loss: 0.6415
Epoch 3/5
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.6822 - loss: 0.6302 - val_accuracy: 0.7296 - val_loss: 0.6085
Epoch 4/5
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.7817 - loss: 0.5798 - val_accuracy: 0.7335 - val_loss: 0.5775
Epoch 5/5
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.8238 - loss: 0.5237 - val_accuracy: 0.7490 - val_loss: 0.5543


In [105]:
# Test
test_loss, test_acc = model.evaluate(X_test_seq, np.array(y_test))
print("Test Accuracy:", test_acc)

14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 346us/step - accuracy: 0.7862 - loss: 0.5627
Test Accuracy: 0.7803738117218018
